[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AndresOrdonez369/deepLearningWeek1/blob/main/proyectos/NB03_miniproyecto_cooperativa_starter.ipynb)

# Mini-proyecto PBL · Clase 1 — Modelo base de riesgo de mora con redes neuronales
**Neural Networks: Deep Learning** · UIDE / EIG Campus · Docente: **Andrés Ordóñez-Bolaños** · Notebook *starter* (grupal) — versión estudiante

### El caso
La **Cooperativa de Ahorro y Crédito "Valle Andino" (ficticia)** atiende socios en 10 provincias del Ecuador: comerciantes del mercado, agricultores, transportistas, artesanos.
Su comité de crédito quiere una herramienta que estime la **probabilidad de que un socio caiga en mora de más de 30 días** (`mora_30d = 1`) para priorizar el acompañamiento (llamadas, reestructuración, educación financiera) **antes** de que el atraso ocurra.

> ⚠️ Los datos son **100 % sintéticos**, generados para este curso. Cualquier parecido con una institución real es coincidencia. Si tu grupo **ya tiene el dataset de su proyecto de titulación**, úsalo en lugar de este (cambia solo la sección 1) y mantén la misma estructura de experimentos.

### Lo que este notebook ya resuelve vs. lo que hace tu grupo
| Sección | Estado | Mínimo técnico del sílabo |
|---|---|---|
| 1–3 Carga, exploración, partición sin fuga | ✅ dado (completa los `TODO`) | — |
| 4 Líneas base | ✅ dado | — |
| 5 **Perceptrón desde cero** | 🔧 completar | Implementación de un perceptrón |
| 6 **MLP + retropropagación desde cero** | 🔧 completar | Retropropagación en un dataset |
| 7 **MLP en Keras** | 🔧 completar | MLP en Keras/TensorFlow |
| 8 **Experimentos de optimización** | 🔧 completar | Optimización y ajuste de parámetros |
| 9 Evaluación final y conclusiones | ✍️ redactar | Justificación de decisiones |

**Regla de oro:** el conjunto de **prueba (test)** se usa **una sola vez**, al final. Todas las decisiones se toman con **validación**.

### Cómo trabajar en grupo con este notebook
1. Una persona del grupo abre el notebook con el botón **Open in Colab** del repositorio, lo guarda (**Archivo → Guardar una copia en Drive**) y lo comparte (**Compartir**, arriba a la derecha) con los demás.
2. Los datos se descargan solos desde GitHub en la sección 1. Si no hay internet, sube `cooperativa_ficticia_mora.csv` al panel de archivos (icono de carpeta a la izquierda).
3. Ejecuten en orden. Después de cada sección hay un **Checkpoint**: si su resultado se parece, sigan; si no, revisen antes de avanzar.
4. Las celdas con `TODO` son las que completan ustedes. Si ven `NotImplementedError`, es la señal de que falta su código.

In [ ]:
# ===== Setup =====
import os, time, warnings
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2"); warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, accuracy_score, roc_curve

SEED = 2026
rng = np.random.default_rng(SEED); keras.utils.set_random_seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
print("TF", tf.__version__, "| Keras", keras.__version__)

## 1 · Carga de datos
La celda descarga `cooperativa_ficticia_mora.csv` desde el repositorio del curso (carpeta `proyectos/`, junto a este notebook). Si ya está junto al notebook, usa esa copia; si no hay internet, te pide subirlo.

**Diccionario de datos**
| Variable | Tipo | Descripción |
|---|---|---|
| `edad` | entero | Edad del socio (años) |
| `provincia` | categórica | Provincia de la agencia |
| `zona` | categórica | `urbana` / `rural` |
| `sector` | categórica | Actividad económica principal |
| `ingreso_mensual` | USD | Ingreso mensual declarado |
| `monto_credito` | USD | Monto desembolsado |
| `plazo_meses` | entero | Plazo del crédito |
| `cuota_mensual` | USD | Cuota fija (tasa anual ficticia 18 %) |
| `antiguedad_socio_meses` | entero | Meses como socio |
| `creditos_previos` | entero | Créditos anteriores en la cooperativa |
| `atrasos_previos_12m` | entero | Atrasos registrados en los últimos 12 meses |
| `tiene_garante` | 0/1 | Crédito con garante |
| `ahorro_promedio` | USD | Saldo promedio de ahorro |
| **`mora_30d`** | **0/1** | **Objetivo:** cayó en mora > 30 días |

In [ ]:
CSV = "cooperativa_ficticia_mora.csv"
URL = "https://raw.githubusercontent.com/AndresOrdonez369/deepLearningWeek1/main/proyectos/" + CSV
if not os.path.exists(CSV):
    for local in ("proyectos/" + CSV, "../proyectos/" + CSV):   # running from a clone of the repo
        if os.path.exists(local):
            CSV = local; break
    else:
        try:
            pd.read_csv(URL).to_csv(CSV, index=False)    # download from GitHub
            print("Downloaded from GitHub:", URL)
        except Exception:
            try:
                from google.colab import files            # no internet: upload by hand (Colab only)
                print("Upload", CSV); files.upload()
            except ImportError:
                raise FileNotFoundError(f"Put {CSV} next to this notebook.")

df = pd.read_csv(CSV)
TARGET = "mora_30d"
print(df.shape); display(df.head())
print(f"\nPrevalence of mora_30d: {df[TARGET].mean():.2%}")

**Checkpoint:** 8 000 filas, 14 columnas y prevalencia de mora de **12.70 %**.

## 2 · Exploración mínima (orientada a decisiones de modelado)
Buscamos señales de **no linealidad** e **interacciones**: si existen, un MLP tiene algo que aportar sobre una regresión logística.

In [ ]:
df_eda = df.assign(ratio_cuota_ingreso=df.cuota_mensual / df.ingreso_mensual)
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

# (a) default rate vs payment burden (binned) -> is the effect linear?
bins = pd.qcut(df_eda.ratio_cuota_ingreso, 12)
df_eda.groupby(bins, observed=True)[TARGET].mean().plot(ax=ax[0], marker="o")
ax[0].set_title("Tasa de mora vs. cuota/ingreso"); ax[0].set_xlabel("decil de cuota/ingreso"); ax[0].tick_params(axis="x", labelrotation=90, labelsize=7)

# (b) interaction: sector x term
pd.crosstab(df.sector, df.plazo_meses, values=df[TARGET], aggfunc="mean").plot(kind="bar", ax=ax[1], width=.85, legend=True)
ax[1].set_title("Tasa de mora por sector y plazo"); ax[1].legend(title="plazo", fontsize=7, ncol=2)

# (c) age profile
df.groupby(pd.cut(df.edad, range(15, 80, 5)), observed=True)[TARGET].mean().plot(ax=ax[2], marker="o")
ax[2].set_title("Tasa de mora por edad"); ax[2].tick_params(axis="x", labelrotation=90, labelsize=7)
plt.tight_layout(); plt.show()

> **TODO (grupo):** escribe aquí 3 hallazgos de la exploración y qué implican para el modelo (¿hay efectos no lineales? ¿interacciones? ¿qué variable crees que pesa más y por qué?).
>
> 1. …
> 2. …
> 3. …

## 3 · Partición y preprocesamiento **sin fuga de información**
60 % entrenamiento / 20 % validación / 20 % prueba, estratificado por la clase. El escalador y el *one-hot* se **ajustan solo con entrenamiento**.

*Analogía:* calibrar la balanza con las mismas papas que luego vas a vender "para ver si pesa bien" no prueba nada. El conjunto de prueba es el cliente que llega mañana.

In [ ]:
X = df.drop(columns=TARGET); y = df[TARGET].values.astype("float32")
num_cols = list(X.select_dtypes("number").columns); cat_cols = [c for c in X.columns if c not in num_cols]

X_tmp, X_te, y_tmp, y_te = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
X_tr, X_va, y_tr, y_va = train_test_split(X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=SEED)

prep = ColumnTransformer([("num", StandardScaler(), num_cols),
                          ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)])
A_tr = prep.fit_transform(X_tr).astype("float32")      # fit ONLY on train
A_va = prep.transform(X_va).astype("float32")
A_te = prep.transform(X_te).astype("float32")
feat_names = prep.get_feature_names_out()
print("Train/Val/Test:", A_tr.shape, A_va.shape, A_te.shape, "| features:", len(feat_names))

**Checkpoint:** Train/Val/Test = (4800, 28), (1600, 28), (1600, 28).

## 4 · Líneas base y métricas
Con 12–13 % de mora, un modelo que **siempre dice "no mora"** tiene ~87 % de *accuracy*… y no sirve para nada. Por eso reportamos:
- **ROC-AUC**: probabilidad de que un moroso al azar reciba mayor puntaje que un no moroso al azar (independiente del umbral).
- **PR-AUC** (*average precision*): más informativa cuando la clase positiva es rara (su línea base es la prevalencia).
- **Recall / precisión** en un umbral de negocio.

In [ ]:
def evaluate(name, p, y_true, thr=0.5):
    pred = (p >= thr).astype(int)
    return {"modelo": name, "ROC-AUC": roc_auc_score(y_true, p), "PR-AUC": average_precision_score(y_true, p),
            "accuracy": accuracy_score(y_true, pred), "recall": recall_score(y_true, pred, zero_division=0),
            "precision": precision_score(y_true, pred, zero_division=0), "umbral": thr}

results_val = []
results_val.append(evaluate("Siempre 'no mora'", np.zeros_like(y_va), y_va))
logreg = LogisticRegression(max_iter=3000).fit(A_tr, y_tr)
p_lr_va = logreg.predict_proba(A_va)[:, 1]
results_val.append(evaluate("Regresión logística", p_lr_va, y_va))
pd.DataFrame(results_val).round(3)

**Checkpoint:** "Siempre no mora" tiene accuracy 0.873 pero ROC-AUC 0.500 (no sirve); la regresión logística llega a ROC-AUC ≈ 0.85.

## 5 · Parte A — Perceptrón de Rosenblatt desde cero
Etiquetas $y\in\{-1,+1\}$; actualización solo en errores: $\mathbf{w}\leftarrow\mathbf{w}+\eta\,y\,\tilde{\mathbf{x}}$.
Como estos datos **no son linealmente separables**, el perceptrón clásico no converge: implementa también la variante ***pocket*** (guarda el mejor $\mathbf{w}$ visto).

**TODO (grupo):** completa la función `perceptron_train`. Usa como puntaje $s=\mathbf{w}^\top\tilde{\mathbf{x}}$ (sirve para ROC-AUC aunque no sea una probabilidad).

In [ ]:
def perceptron_train(X, y01, eta=1.0, epochs=20, seed=SEED):
    # Classic perceptron + pocket. Returns (w_last, w_pocket, mistakes_per_epoch).
    r = np.random.default_rng(seed)
    Xt = np.c_[X, np.ones(len(X))]            # augmented input (bias inside w)
    y = np.where(y01 > 0, 1, -1)
    w = np.zeros(Xt.shape[1]); w_best, best_err = w.copy(), np.inf
    mistakes = []
    for ep in range(epochs):
        errs = 0
        for i in r.permutation(len(Xt)):
            # TODO: if sample i is misclassified (y_i * w·x_i <= 0), update w and count the error
            pass
        mistakes.append(errs)
        train_err = np.mean(np.sign(Xt @ w + 1e-12) != y)   # pocket: keep best weights
        if train_err < best_err:
            best_err, w_best = train_err, w.copy()
    return w, w_best, mistakes

w_last, w_pocket, mistakes = perceptron_train(A_tr, y_tr)
score = lambda w, X: np.c_[X, np.ones(len(X))] @ w
plt.figure(figsize=(6, 3)); plt.plot(mistakes, "o-"); plt.xlabel("época"); plt.ylabel("errores")
plt.title("El perceptrón no llega a 0 errores: datos no separables"); plt.show()
results_val.append(evaluate("Perceptrón (pocket)", 1/(1+np.exp(-score(w_pocket, A_va))), y_va))
pd.DataFrame(results_val).round(3)

**Checkpoint:** los errores por época bajan y suben sin llegar a 0 (los datos no son separables); el perceptrón *pocket* queda cerca de ROC-AUC 0.83.

## 6 · Parte B — MLP con retropropagación desde cero (NumPy)
Ecuaciones (filas = ejemplos, $\Delta^{[L]}=(\hat Y-Y)/B$):
$$\nabla_{W^{[l]}}\mathcal{L}=A^{[l-1]\top}\Delta^{[l]},\quad \nabla_{\mathbf b^{[l]}}\mathcal{L}=\mathbf 1^\top\Delta^{[l]},\quad \Delta^{[l-1]}=(\Delta^{[l]}W^{[l]\top})\odot\varphi'(Z^{[l-1]}).$$
**TODO (grupo):** (1) completa `backward`; (2) aprueba el *gradient check* (error relativo $<10^{-6}$); (3) entrena y reporta.

In [ ]:
class NumpyMLP:
    def __init__(self, sizes, seed=SEED):
        r = np.random.default_rng(seed)
        self.W = [r.normal(0, np.sqrt(2 / i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]   # He init
        self.b = [np.zeros(o) for o in sizes[1:]]

    def forward(self, X):
        self.Z, self.A = [], [X]
        for l, (W, b) in enumerate(zip(self.W, self.b)):
            Z = self.A[-1] @ W + b
            A = 1 / (1 + np.exp(-Z)) if l == len(self.W) - 1 else np.maximum(0, Z)   # ReLU hidden, sigmoid out
            self.Z.append(Z); self.A.append(A)
        return self.A[-1]

    def loss(self, X, Y, eps=1e-12):
        P = self.forward(X); return -np.mean(Y*np.log(P+eps) + (1-Y)*np.log(1-P+eps))

    def backward(self, Y):
        # TODO: implement BP1-BP4 and return (gW, gb), lists of gradients per layer
        raise NotImplementedError("Complete backward()")

def grad_check(model, X, Y, eps=1e-6):
    model.forward(X); gW, gb = model.backward(Y)
    ana = np.concatenate([g.ravel() for g in gW + gb]); num = []
    for P in model.W + model.b:
        for i in np.ndindex(P.shape):
            old = P[i]; P[i] = old + eps; lp = model.loss(X, Y); P[i] = old - eps; lm = model.loss(X, Y); P[i] = old
            num.append((lp - lm) / (2*eps))
    num = np.array(num); return np.linalg.norm(num-ana) / (np.linalg.norm(num) + np.linalg.norm(ana))

d = A_tr.shape[1]
# Gradient check with small NON-zero biases: with b = 0 a dead layer can produce Z = 0 exactly,
# where ReLU is not differentiable and the finite difference "sees" the kink (false alarm).
gc_model = NumpyMLP([d, 8, 4, 1])
for bias in gc_model.b: bias += 0.1
gc = grad_check(gc_model, A_tr[:32].astype(float), y_tr[:32, None].astype(float))
print(f"Gradient check relative error: {gc:.2e}  ->", "OK ✅" if gc < 1e-6 else "REVIEW ❌")

In [ ]:
def fit_numpy(model, X, Y, Xv, Yv, epochs=60, lr=0.005, batch=64, momentum=0.9):
    vW = [np.zeros_like(W) for W in model.W]; vb = [np.zeros_like(b) for b in model.b]
    hist = {"train": [], "val": [], "val_auc": []}
    for ep in range(epochs):
        idx = rng.permutation(len(X))
        for s in range(0, len(X), batch):
            j = idx[s:s+batch]; model.forward(X[j]); gW, gb = model.backward(Y[j])
            for l in range(len(model.W)):                      # SGD + momentum
                vW[l] = momentum*vW[l] + gW[l]; model.W[l] -= lr*vW[l]
                vb[l] = momentum*vb[l] + gb[l]; model.b[l] -= lr*vb[l]
        hist["train"].append(model.loss(X, Y)); hist["val"].append(model.loss(Xv, Yv))
        hist["val_auc"].append(roc_auc_score(Yv.ravel(), model.forward(Xv).ravel()))
    return hist

np_mlp = NumpyMLP([d, 32, 16, 1])
h_np = fit_numpy(np_mlp, A_tr.astype(float), y_tr[:, None], A_va.astype(float), y_va[:, None])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h_np["train"], label="train"); ax[0].plot(h_np["val"], label="val"); ax[0].set_title("BCE — MLP NumPy"); ax[0].legend()
ax[1].plot(h_np["val_auc"]); ax[1].set_title("ROC-AUC validación"); plt.tight_layout(); plt.show()
results_val.append(evaluate("MLP NumPy 32-16 (SGD+mom)", np_mlp.forward(A_va.astype(float)).ravel(), y_va))
pd.DataFrame(results_val).round(3)

**Checkpoint:** gradient check < 1e-6 ("OK") y un MLP NumPy con ROC-AUC de validación ≈ 0.87.

## 7 · Parte C — El mismo MLP en Keras
Usamos `EarlyStopping` sobre la AUC de validación como "freno de mano" (lo formalizaremos en la Clase 3: regularización y generalización).

**TODO (grupo):** justifica en el informe el número de capas/neuronas, la activación, la inicialización y el optimizador elegidos (¡con argumentos de la clase!).

In [ ]:
def build_mlp(d, hidden=(32, 16), lr=1e-3, optimizer="adam", momentum=0.0):
    keras.utils.set_random_seed(SEED)
    inp = keras.Input(shape=(d,), name="member_features")
    h = inp
    for k, units in enumerate(hidden):
        h = keras.layers.Dense(units, activation="relu", kernel_initializer="he_normal", name=f"hidden_{k+1}")(h)
    out = keras.layers.Dense(1, activation="sigmoid", name="p_mora")(h)
    model = keras.Model(inp, out, name="mlp_mora")
    opt = {"adam": keras.optimizers.Adam(lr), "sgd": keras.optimizers.SGD(lr, momentum=momentum)}[optimizer]
    model.compile(optimizer=opt, loss="binary_crossentropy", metrics=[keras.metrics.AUC(name="auc")])
    return model

mlp = build_mlp(d); mlp.summary()
es = keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True)
hist = mlp.fit(A_tr, y_tr, validation_data=(A_va, y_va), epochs=80, batch_size=64, callbacks=[es], verbose=0)
print("Epochs run:", len(hist.history["loss"]))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="val"); ax[0].set_title("BCE — Keras"); ax[0].legend()
ax[1].plot(hist.history["auc"], label="train"); ax[1].plot(hist.history["val_auc"], label="val"); ax[1].set_title("ROC-AUC — Keras"); ax[1].legend()
plt.tight_layout(); plt.show()
p_mlp_va = mlp.predict(A_va, verbose=0).ravel()
results_val.append(evaluate("MLP Keras 32-16 (Adam)", p_mlp_va, y_va))
pd.DataFrame(results_val).round(3)

**Checkpoint:** el MLP en Keras tiene 1 473 parámetros y llega a ROC-AUC de validación ≈ 0.87.

## 8 · Parte D — Experimentos de optimización (diseño factorial pequeño)
Variamos **optimizador × tasa de aprendizaje × tamaño de batch**, con el mismo modelo, misma semilla y misma partición. Seleccionamos por **AUC de validación**.

**TODO (grupo):** agrega al menos **una dimensión propia** (p. ej. ancho/profundidad, activación `tanh`, inicialización) y analiza la tabla: ¿qué factor importa más? ¿hay interacción entre $\eta$ y el batch?

In [ ]:
grid = [(opt, lr, bs) for opt in ["sgd", "sgd_mom", "adam"] for lr in [1e-3, 1e-2] for bs in [32, 128]]
rows = []
for opt, lr, bs in grid:
    m = build_mlp(d, lr=lr, optimizer="sgd" if opt.startswith("sgd") else "adam", momentum=0.9 if opt == "sgd_mom" else 0.0)
    t0 = time.time()
    h = m.fit(A_tr, y_tr, validation_data=(A_va, y_va), epochs=25, batch_size=bs, verbose=0)
    rows.append({"optimizador": opt, "lr": lr, "batch": bs, "val_auc_final": h.history["val_auc"][-1],
                 "val_auc_max": max(h.history["val_auc"]), "segundos": round(time.time() - t0, 1)})
exp = pd.DataFrame(rows).sort_values("val_auc_max", ascending=False).reset_index(drop=True)
exp.round(4)

In [ ]:
pivot = exp.pivot_table(index="optimizador", columns=["lr", "batch"], values="val_auc_max")
plt.figure(figsize=(8, 2.8)); plt.imshow(pivot.values, cmap="viridis", aspect="auto")
plt.xticks(range(pivot.shape[1]), [f"lr={a}\nB={b}" for a, b in pivot.columns], fontsize=8); plt.yticks(range(3), pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]; plt.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=9,
                 color="k" if v > np.nanmean(pivot.values) else "w")
plt.colorbar(label="AUC val (máx)"); plt.title("Mapa de experimentos"); plt.tight_layout(); plt.show()

## 9 · Evaluación final (una sola vez) y elección del umbral
1. Elige **un** modelo final con base en validación.
2. Elige el umbral de decisión con validación según el objetivo de negocio (ej.: detectar al menos el 70 % de los futuros morosos).
3. Reporta en **test** — y no vuelvas a tocar el modelo después de mirar estos números.

In [ ]:
# Business rule (discuss it in your report): each model gets the threshold that reaches
# >= 70% recall on VALIDATION, so all models are compared at the same operating point.
def thr_for_recall(y_true, p, target=0.70):
    fpr, tpr, thr = roc_curve(y_true, p)
    return float(thr[np.argmax(tpr >= target)])

models = {
    "Regresión logística": (lambda A: logreg.predict_proba(A)[:, 1]),
    "Perceptrón (pocket)": (lambda A: 1/(1+np.exp(-score(w_pocket, A)))),
    "MLP NumPy 32-16":     (lambda A: np_mlp.forward(A.astype(float)).ravel()),
    "MLP Keras 32-16":     (lambda A: mlp.predict(A, verbose=0).ravel()),
}
final = []
for name, f in models.items():
    t = thr_for_recall(y_va, f(A_va))            # chosen on validation
    final.append(evaluate(name, f(A_te), y_te, t))  # reported once on test
pd.DataFrame(final).round(3)

### Preguntas guía para el informe (pensamiento crítico)
1. ¿Cuánto mejora el MLP sobre la regresión logística? ¿Es una mejora **relevante para el negocio** o ruido? (pista: repite con 3 semillas y reporta media ± desviación).
2. **Reto:** crea la variable `ratio_cuota_ingreso` y vuelve a entrenar la regresión logística. ¿Se cierra la brecha? ¿Qué dice esto sobre *feature engineering* vs. *representation learning*?
3. ¿Por qué el perceptrón no converge aquí? Relaciónalo con el teorema de Novikoff.
4. ¿Qué umbral usarías si cada llamada preventiva cuesta USD 2 y cada crédito en mora cuesta en promedio USD 400 en provisiones y cobranza? Plantea la función de costo.
5. **Ética (adelanto de la Clase 3):** `provincia` y `zona` pueden actuar como *proxies* de grupos vulnerables. ¿Deberían estar en el modelo? ¿Cómo lo verificarías?
6. **Conexión con tu proyecto de titulación:** ¿qué parte de este *pipeline* reutilizarás tal cual y qué cambia con tus datos?

> ✍️ **Conclusiones del grupo:** …